In [ ]:
# ==============================================
# Step 1: Mount Google Drive
# ==============================================
from google.colab import drive
drive.mount('/content/drive')

# ==============================================
# Step 2: Import libraries
# ==============================================
import os
import pandas as pd
from ftplib import FTP
import time
from urllib.parse import urlparse

# ==============================================
# Step 3: Load CSV file from Google Drive
# ==============================================
# Replace this with the actual path of your CSV in Drive
csv_path = '/content/drive/MyDrive/phd_experiment_2/NCBI_DATA_DOWNLOAD/missing_bacteria_for_download_2.csv'

df = pd.read_csv(csv_path)
print("CSV loaded successfully!")
print(df.head())

# ==============================================
# Step 4: Base folder to save downloads
# ==============================================
base_save_dir = '/content/drive/MyDrive/phd_experiment_2/Genomic_Data'
os.makedirs(base_save_dir, exist_ok=True)

# ==============================================
# Step 5: Function to download .txt, .tsv, .gz files from FTP
# ==============================================
def download_ftp_files(organism, ftp_url, retries=3, delay=5):
    parsed_url = urlparse(ftp_url)
    ftp_server = parsed_url.hostname
    ftp_path = parsed_url.path

    if not ftp_server:
        print(f"Invalid FTP URL for {organism}: {ftp_url}")
        return

    folder_path = os.path.join(base_save_dir, organism.replace(" ", "_"))
    os.makedirs(folder_path, exist_ok=True)

    attempt = 0
    while attempt < retries:
        try:
            print(f" Connecting to FTP server: {ftp_server}")
            with FTP(ftp_server) as ftp:
                ftp.login() # Anonymous login

                print(f" Changing directory to: {ftp_path}")
                ftp.cwd(ftp_path)

                file_list = ftp.nlst() # List files in the current directory
                print(f"Found {len(file_list)} items in the directory.")

                # Filter files ending with .txt, .tsv, or .gz
                download_files = [f for f in file_list if f.endswith(('.txt', '.tsv', '.gz'))]

                if not download_files:
                    print(f" No .txt/.tsv/.gz files found for {organism} in {ftp_path}")
                    return

                print(f"Attempting to download {len(download_files)} files.")
                for i, filename in enumerate(download_files, 1):
                    local_filepath = os.path.join(folder_path, filename)
                    try:
                        with open(local_filepath, 'wb') as local_file:
                            ftp.retrbinary(f"RETR {filename}", local_file.write)
                        print(f" {organism}: Downloaded ({i}/{len(download_files)}) - {filename}")
                    except Exception as e:
                        print(f" Failed to download {filename} for {organism}: {e}")
                return # Exit after successful download

        except Exception as e:
            attempt += 1
            print(f" Attempt {attempt} failed for {organism}: {e}")
            if attempt < retries:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f" Failed to download files for {organism} after {retries} attempts.")

# ==============================================
# Step 6: Loop through each organism and download files
# ==============================================
# Define the starting index for the loop to resume from where it was interrupted
start_index = 7 # Based on the last processed 'idx' in the output

for idx, row in df.iloc[start_index:].iterrows():
    organism = str(row['genus_and_species']).strip()
    ftp_url = str(row['ftp_path']).strip()

    print(f"\n Processing organism: {organism}")
    download_ftp_files(organism, ftp_url)

print("\n All files have been processed!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ CSV loaded successfully!
               genus_and_species  taxonomy_id strain superkingdom  \
0  Algoriphagus machipongonensis       388413    PR1     Bacteria   
1               Ruegeria conchae       981384   TW15     Bacteria   
2         Aquimarina agarivorans       980584   HQM9     Bacteria   
3            Ochrovirga pacifica      1042376    S85     Bacteria   
4         Lentibacillus jeotgali       558169   Grbi     Bacteria   

           phylum                class             order             family  \
0   Bacteroidetes           Cytophagia      Cytophagales  Cyclobacteriaceae   
1  Proteobacteria  Alphaproteobacteria   Rhodobacterales   Rhodobacteraceae   
2   Bacteroidetes       Flavobacteriia  Flavobacteriales  Flavobacteriaceae   
3   Bacteroidetes       Flavobacteriia  Flavobacteriales  Flavobacteriaceae   
4      Firmicutes              Bac